<p class="lead">Every project in this laboratory — a QSAR model of kinase inhibitors, a virtual-screening campaign, a molecular-dynamics analysis, a gene-expression comparison — ends in the same place: a table of numbers, a figure, and a claim that the numbers support the figure. This unit builds the instrument that produces all three, in R: an environment you understand and can rebuild, vectors and data frames used as practitioners use them, a document a second person can knit from scratch and obtain your numbers, the group's project layout, and an explicit policy for working with an AI coding assistant.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad construye el entorno de trabajo del laboratorio en R: el modelo de vectores atómicos (tipo, longitud, atributos y la semántica de copia al modificar, que distingue a R de NumPy), la vectorización medida contra bucles — incluido el error clásico de hacer crecer un vector dentro del bucle —, el reciclado de longitudes y su error silencioso, y las semillas aleatorias explícitas. Después las tablas: <code>data.frame</code>, <code>tibble</code> y <code>data.table</code>, indexación por nombre y por posición, tipos de dato y memoria, factores frente a cadenas de texto, y el patrón dividir–aplicar–combinar en <code>dplyr</code> y en <code>data.table</code>. Fijamos la disciplina del documento R-Markdown — ejecución lineal, <em>knit</em> desde cero antes de guardar, ningún estado oculto — y el momento en que un cuaderno debe convertirse en un script dentro de <code>R/</code>. Creamos la plantilla de proyecto del grupo, con <code>renv</code> en lugar de conda para fijar el entorno, y establecemos una política honesta para los asistentes de inteligencia artificial: el asistente redacta; el estudiante lee, prueba, explica y atribuye. Los datos son un extracto de 200 filas de la tabla ChEMBL del curso, usado solo para practicar la carga e inspección.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Set up a working R environment locally (a project plus <code>renv</code>) and in Google Colab, and explain what an environment is and why there is one per project.</li>
<li>Use R's atomic vectors and matrices fluently: type, length and attributes, copy-on-modify semantics, vectorised operations measured against loops, length recycling, and random generators with explicit seeds.</li>
<li>Use tabular R fluently: <code>data.frame</code>, <code>tibble</code> and <code>data.table</code>, name- and position-based indexing, storage types and memory, factors and character columns, and basic input/output.</li>
<li>Write an R-Markdown document that someone else can knit from a clean session and obtain your numbers, and recognise when a document should become a script in <code>R/</code>.</li>
<li>Create and justify the group's project template, and place every file a project produces in it without hesitation.</li>
<li>Use an AI coding assistant responsibly: draft, read, test, explain, attribute — and never keep code you cannot explain.</li>
<li>Describe, in one sentence each, the stages of a thesis project in this group and the tool that serves each stage.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it once linearly, running each chunk as you reach it and reading its output before moving on; then re-run chunks while changing <em>one</em> thing at a time (a vector length, a storage type, a seed). Boxes carry fixed meanings throughout the course: <strong>key</strong> boxes state something to remember, <strong>warning</strong> boxes a mistake routinely made in published work, <strong>Domain context</strong> boxes define a chemistry or simulation term for readers who come from computing, and <strong>Bridge</strong> boxes point to the unit or sister course that continues the thread. In a fresh Colab R runtime the packages used here are installed once with <code>install.packages(c("data.table","tibble","microbenchmark"))</code>; <code>dplyr</code> and <code>ggplot2</code> are already present. The data file is read through the <code>data_path</code> helper of the setup chunk, which finds the frozen copy in <code>../data/</code> and downloads it once otherwise. This is the <strong>R edition</strong> of the unit; its <a href="../../Python/U1_Toolkit/index.html">Python counterpart</a> covers the same material with NumPy and pandas, and every number below was produced by the R code shown here, not copied from there.</p></div>



In [ ]:
## Setup: the whole stack used in this unit, plus the course data loader.
suppressPackageStartupMessages({
  library(data.table)     # fast I/O and a concise table grammar
  library(dplyr)          # the tidyverse table verbs (filter() masks stats::filter)
  library(tibble)
  library(ggplot2)
  library(microbenchmark) # reliable sub-millisecond timing
})

set.seed(0)               # the global seed; every resampling below sets its own as well

PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9.5) +
          theme(plot.title = element_text(face = "bold", size = 10.5),
                panel.grid.minor = element_blank()))

REPO <- "DanielBustosG/DanielBustosG.github.io"       # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- sprintf("https://raw.githubusercontent.com/%s/main/DataScience/data/", REPO)

data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dir.create("data", showWarnings = FALSE)
  dest <- file.path("data", name)
  if (!file.exists(dest)) download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  dest
}
load_data <- function(name, ...) data.table::fread(data_path(name), ...)

cat(R.version.string, "on", R.version$platform, "\n")
for (p in c("data.table", "dplyr", "ggplot2", "tibble", "microbenchmark"))
  cat(sprintf("%-15s %s\n", p, as.character(packageVersion(p))))
cat("data file found:", file.exists(data_path("rock1_chembl_raw.csv")), "\n")



The output above is the first thing any reader of your document should see, and the first thing to check when a result cannot be reproduced: which interpreter ran the code, and which package versions were in effect. Unit 2 turns this habit into a formal record of provenance; for now, accept the rule: **a document prints its own environment before it prints anything else**.

## 1. What the toolkit is for

A thesis project in the group moves through five stages. It begins with a **question** that must be made quantitative — "is this scaffold more potent than that one?" becomes "is the median pActivity of the 30 compounds containing scaffold A greater than that of the 45 containing scaffold B, and how wide is the uncertainty on that difference?". It continues with **data**: a download from a public database, an output folder from a docking campaign, a trajectory from a simulation, a count matrix from a sequencing core. Then comes a **model or a simulation**, then **validation**, which is where most projects are won or lost, and finally a **paper**, in which every number must be traceable to a file and every figure to a script.

R serves all five stages, which is why this laboratory keeps an R edition of its entire analytical practice alongside the Python one. R was designed from the start as a language for data analysis and graphics<sup class="c"><a href="#r1">[1]</a></sup>, so the things this course does constantly — fit a model, summarise by group, draw a figure with its uncertainty — are one function call rather than an assembled pipeline. The modern working layer is the **tidyverse**, a set of packages that share a data representation and a grammar<sup class="c"><a href="#r2">[2]</a></sup>: `dplyr` for the table verbs, `tidyr` for reshaping, `readr` for input, `purrr` for iteration, `ggplot2` for figures. Two ideas from that family are worth naming now because the rest of the course assumes them: **tidy data**, one row per observation and one column per variable<sup class="c"><a href="#r5">[5]</a></sup>, and the **split–apply–combine** pattern, which turns "summarise each group" into a single declarative statement<sup class="c"><a href="#r3">[3]</a></sup>. Figures follow a **layered grammar of graphics**<sup class="c"><a href="#r4">[4]</a></sup> — data, a mapping from variables to visual properties, and geometric layers — which is why a `ggplot2` figure can be modified by adding to it rather than by redrawing it.

Alongside the tidyverse sits `data.table`, which solves a different problem: very large tables, aggregated in place with minimal copying. The two are not rivals, and this course uses both. Around that core sit the domain packages the group depends on: **Bioconductor** for genomics<sup class="c"><a href="#r9">[9]</a></sup>, with `limma` for differential expression; `ChemmineR` for compound mining<sup class="c"><a href="#r10">[10]</a></sup>; `bio3d` for protein structures and trajectories<sup class="c"><a href="#r11">[11]</a></sup>; `seqinr` for sequences<sup class="c"><a href="#r12">[12]</a></sup>. Because they all accept and return R vectors, matrices and data frames, a descriptor column can be scaled by base R, modelled by `glmnet` and plotted by `ggplot2` without a single conversion function written by you. That interoperability, not any individual package, is the thing worth learning.

<div class="box bridge"><h4>Domain context — what a bioactivity record is</h4>
<p>The data file used in this unit is a table of <strong>bioactivity records</strong>. One record says: a particular <em>compound</em> (a small organic molecule, identified by a database code and by a text string that encodes its structure) was tested against a particular <em>target</em> (here the human enzyme ROCK1, a protein kinase studied in this laboratory for glaucoma and fibrosis indications<sup class="c"><a href="#r15">[15]</a></sup>) in a particular <em>assay</em> (a laboratory experiment described in a publication), and the measured potency was a certain concentration. The concentration is usually an <strong>IC50</strong> — the concentration of compound that halves the target's activity — or a <strong>Ki</strong>/<strong>Kd</strong>, which are binding constants. All three are concentrations, so <em>smaller means more potent</em>, and they span many orders of magnitude, which is why they are almost always analysed on a logarithmic scale (Section 5). The records in <code>rock1_chembl_raw.csv</code> were downloaded from ChEMBL<sup class="c"><a href="#r16">[16]</a></sup>, a public database that curates such measurements out of the medicinal-chemistry literature. For this unit that is all the chemistry you need: a record is a row, a compound is an identifier plus a structure string, and a potency is a positive number in nanomolar units whose logarithm is the quantity we model. One warning to carry forward: two laboratories measuring the same compound against the same target routinely disagree by half a log unit or more<sup class="c"><a href="#r17">[17]</a></sup>, so the numbers in this table are measurements, not constants of nature.</p></div>



In [ ]:
## A schematic of the project lifecycle and the tool that serves each stage.
stages <- c("question", "data", "model /\nsimulation", "validation", "paper")
tools  <- c("R Markdown +\nliterature", "data.table, APIs,\nSQL (U3)", "glmnet, ranger,\nbio3d, limma",
            "resampling,\nstatistics (U8-U12)", "ggplot2, tables,\nrepository (U23-U24)")
d <- data.frame(x = seq_along(stages), stage = stages, tool = tools)

p <- ggplot(d, aes(x = x)) +
  geom_tile(aes(y = 0.76), width = 0.84, height = 0.42,
            fill = "#dce8f5", colour = PAL[["blue"]], linewidth = 0.5) +
  geom_text(aes(y = 0.76, label = stage), fontface = "bold", size = 3.1) +
  geom_text(aes(y = 0.33, label = tool), size = 2.5, colour = "grey25") +
  geom_segment(data = d[1:4, ], aes(x = x + 0.44, xend = x + 0.54, y = 0.76, yend = 0.76),
               arrow = arrow(length = unit(0.08, "cm"), type = "closed"), colour = PAL[["blue"]]) +
  geom_segment(aes(x = 5.42, xend = 0.58, y = 1.08, yend = 1.08), linetype = "dotted",
               colour = PAL[["orange"]],
               arrow = arrow(length = unit(0.08, "cm"), type = "closed")) +
  annotate("text", x = 3, y = 1.16, size = 2.7, colour = PAL[["orange"]],
           label = "every stage can send you back to an earlier one") +
  scale_x_continuous(limits = c(0.3, 5.7)) + scale_y_continuous(limits = c(0.1, 1.28)) +
  labs(title = "Stages of a project in the group, and the tool that serves each") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 1.</b> The five stages of a project in this laboratory and the tooling each one uses. The dotted arrow is what beginners underestimate: validation routinely sends a project back to the data, and the data stage back to the question. A toolkit is judged by how cheap it makes those return trips — which is what reproducible project structure (Section 7) and version control (Unit 2) buy you.</p>

### 1.1 Environments: one per project, and why

An **environment**, in the sense this course uses the word, is an isolated set of package versions bound to one project. When you type `library(dplyr)`, R searches a library path; the point of one environment per project is that two projects can then disagree, so a thesis finished last year keeps the `dplyr` version its figures depend on while today's project uses the current one.

In R the instrument for this is **`renv`**, and it replaces the conda environment used by the Python edition of this course. Three commands are the whole workflow. `renv::init()` creates a project-local library and a lock file; `renv::snapshot()` records the exact version (and source) of every package the project actually loads into `renv.lock`; `renv::restore()` rebuilds that library on another machine from the lock file. The lock file is committed to the repository; the library itself is not. Note what `renv` does *not* do: it does not pin the R version itself, the compiler, or the system libraries behind packages such as `arrow` — which is why Unit 2 adds a full environment record and Unit 25 adds containers.

`renv` is not installed in the environment that builds these pages, so the chunk below is shown but not executed; install it once per machine from the console and then run these lines in your own project. Everything else in this unit runs.



In [ ]:
## The renv workflow. Run these in your own project; not evaluated when this page is built.
renv::init()                        # project-local library + renv.lock, in the project root
renv::snapshot()                    # after installing what the project needs: record exact versions
renv::status()                      # does the library still match the lock file?
renv::restore()                     # on a collaborator's machine: rebuild the library from renv.lock



- **`renv` + a project directory** (laboratory workstations; the way this group works). The project carries `renv.lock`; a collaborator runs `renv::restore()` and obtains the same package versions. This is the default for every analysis in the group.
- **Google Colab with an R runtime** (for reading this course). Colab provides a disposable environment that already contains base R, the tidyverse and `ggplot2`; extra packages are installed at the top of the notebook. Every unit has an "Open in Colab (R)" button and a markdown note before the setup chunk whenever an install line is needed.
- **Containers** (Unit 25) when even the R version and the system libraries must be fixed.

The failure mode this prevents has a name in the literature: *"it works on my machine"* is not a communication problem, it is a **data-integrity problem**<sup class="c"><a href="#r18">[18]</a></sup>. If a result depends on an unrecorded package version, the result is not a property of the data and the method; it is a property of a laptop.

## 2. R aimed at data

Data work uses a narrow, recognisable subset of R; this section is a refresher on that subset — the vocabulary the rest of the course writes in.



In [ ]:
## The data-oriented R subset: vectors, logical subsetting, tables, small functions, the pipe.
raw_units <- c("nM", "uM", "nM", "mM", "nM", "uM", NA, "nM")

## 1. Logical subsetting replaces the accumulate-in-a-loop pattern.
present <- raw_units[!is.na(raw_units)]

## 2. table() is a frequency table written as what it is rather than how it is built.
counts <- table(present)

## 3. A small function with a guard and a comment: the unit of code that becomes R/ later.
TO_NANOMOLAR <- c(nM = 1, uM = 1e3, mM = 1e6, M = 1e9, pM = 1e-3)

to_nanomolar <- function(value, unit) {
  # Convert a concentration to nanomolar. Stops for an unknown unit.
  stopifnot(length(unit) == 1, unit %in% names(TO_NANOMOLAR))
  value * TO_NANOMOLAR[[unit]]
}

## 4. sprintf() for readable output; file.path() for every filesystem operation.
cat(sprintf("%d unit strings, %d non-missing, distinct: %s\n",
            length(raw_units), length(present), paste(sort(unique(present)), collapse = ", ")))
cat("counts per unit   :", paste(names(counts), counts, sep = "=", collapse = "  "), "\n")
cat(sprintf("2.5 uM            = %s nM\n", format(to_nanomolar(2.5, "uM"), big.mark = ",")))
cat("data folder exists:", dir.exists(file.path("..", "data")), "\n")
cat("the pipe keeps the reading order:",
    raw_units |> na.omit() |> unique() |> sort() |> paste(collapse = " < "), "\n")



Five constructs carry most R data code. **Logical subsetting** (`x[!is.na(x)]`) replaces the accumulate-in-a-loop pattern. **`table()`** and its relatives answer "how many of each?" in one call. **Small functions with a guard and a comment** are the atoms that later migrate into `R/`; `stopifnot()` is the cheapest contract you can write, and it fails loudly at the moment of the mistake rather than three steps later. **`sprintf()`** and `format()` make output readable — note the `big.mark` argument that inserted the thousands separator above. **`file.path()`** replaces string concatenation for paths: `file.path("..", "data", name)` is correct on every operating system, whereas `paste0("../data/", name)` is a bug waiting for a collaborator on another platform. The **native pipe** `|>` is a reading aid: it puts the operations in the order they happen instead of inside-out.

The sixth construct is error handling, and it deserves its own chunk, because the single most common accident in data work is a conversion that fails silently on a value you did not know was in the column.



In [ ]:
## Conditions: fail loudly on bad input, and distinguish 'missing' from 'wrong'.
to_nanomolar_safe <- function(value, unit) {
  # Convert to nM; return (NA, reason) for a missing or unrecognised input.
  if (length(value) == 0 || is.na(value))
    return(list(nm = NA_real_, status = "missing value"))
  if (!unit %in% names(TO_NANOMOLAR))
    return(list(nm = NA_real_, status = sprintf("unknown unit '%s'", unit)))
  num <- withCallingHandlers(suppressWarnings(as.numeric(value)), error = function(e) NA_real_)
  if (is.na(num))
    return(list(nm = NA_real_, status = sprintf("non-numeric value '%s'", value)))
  list(nm = num * TO_NANOMOLAR[[unit]], status = "ok")
}

trials <- list(list(25, "nM"), list(2.5, "uM"), list(1, "mM"),
               list(10, "10'-8M"), list("n.d.", "nM"), list(NA, "nM"))
for (tr in trials) {
  res <- to_nanomolar_safe(tr[[1]], tr[[2]])
  shown <- if (is.na(res$nm)) "NA" else format(res$nm, big.mark = ",", scientific = FALSE)
  cat(sprintf("%6s %-8s -> %12s nM   %s\n", as.character(tr[[1]]), tr[[2]], shown, res$status))
}



The table printed above is a conversion log, not just a result. The exotic unit string `10'-8M` — which really occurs in the raw ChEMBL table of this course — and the text value `n.d.` were rejected with a reason, and the missing value was distinguished from the wrong ones. **A pipeline that returns `NA` without saying why hides its own curation decisions**; in Unit 6 this log becomes a formal curation table with one row per exclusion step.

<div class="box warn"><h4>A bare <code>try()</code> is a bug</h4>
<p>Writing <code>try(expr, silent = TRUE)</code> and ignoring the result swallows everything, including the interrupt you press to stop a runaway chunk and the memory error that explains why your results are wrong. Handle the specific conditions you expect with <code>tryCatch(expr, error = ..., warning = ...)</code> and let the rest reach you. R's defaults make this especially important because <code>as.numeric("n.d.")</code> returns <code>NA</code> with a <em>warning</em> rather than an error — so a pipeline that suppresses warnings globally converts bad data into missing data without a trace. The one legitimate broad catch in this course is the network call in Unit 3, where a failure of any kind must fall back to a frozen file; even there the condition's message is printed rather than swallowed.</p></div>

## 3. Vectors: the unit of thought

R's atomic vector is a contiguous block of values of one type, plus **attributes** — optional metadata such as `names`, `dim` or `class` — carried alongside it. Three facts explain everything that makes R fast and everything that makes it surprising. First, there are no scalars: `3.1` is a double vector of length one, which is why almost every function in R is vectorised by default. Second, a vector has a single storage type, so mixing types silently coerces all of them to the most general (`c(1, "a")` is character). Third, R has **copy-on-modify** semantics: assigning a vector to a second name does not copy it, but modifying either copy does, so you can never change one object by writing to another.



In [ ]:
## The vector model: type, length, attributes, memory, and copy-on-modify.
a <- matrix(0:11, nrow = 3, byrow = TRUE)          # a matrix IS a vector with a dim attribute
cat("matrix:\n"); print(a)
cat("\ntypeof:", typeof(a), "| length:", length(a), "| dim:", dim(a),
    "| class:", class(a)[1], "\n")
cat("attributes:", paste(names(attributes(a)), collapse = ", "), "\n")
cat("bytes held (integer):", object.size(a), "| as double:",
    object.size(matrix(as.numeric(a), nrow = 3)), "\n")

b <- a                                             # no copy yet: both names point at one object
b[1, 1] <- 999L                                    # the write triggers the copy
cat("\nafter writing into b, a is untouched:", a[1, 1], "| b changed:", b[1, 1], "\n")

f <- function(x) { x[1] <- -1L; x[1] }              # the argument is copied on modification too
cat("a function that modifies its argument returns", f(a), "but a[1] is still", a[1], "\n")
cat("column-major storage: as.vector(a) =", paste(as.vector(a)[1:6], collapse = " "), "...\n")



Four lessons hide in that output. The first is arithmetic on memory: the same twelve numbers occupy roughly half as many bytes as integers as they do as doubles, and on ten million rows that factor decides whether an analysis fits in RAM. The second is that a matrix is not a separate data structure — it is a vector carrying a `dim` attribute, which is why `length(a)` is 12 and not 3. The third is **copy-on-modify**, and it is the single largest difference from the Python edition of this unit: there, slicing a NumPy array returns a *view* that shares memory, so writing into a slice writes into the parent; here, writing into `b` left `a` untouched, and a function cannot modify its caller's data. R trades memory for safety, and the price is that a loop which writes into a large object repeatedly may copy it repeatedly. The fourth is **column-major** storage: `as.vector()` reads down the first column before the second, the opposite of NumPy's default, which matters whenever you reshape a matrix or hand one to a C or Fortran routine.

### 3.1 Vectorisation, measured

"Vectorise your loops" is advice everyone repeats and few have measured. Measure it now, on your own machine, because the size of the effect is the reason the whole R ecosystem is built the way it is. The task is the one this course performs constantly: convert a vector of concentrations in nanomolar to the logarithmic potency scale, \( \mathrm{pActivity} = -\log_{10}(c\,[\mathrm{M}]) = 9 - \log_{10}(c\,[\mathrm{nM}]) \). Three implementations are compared: a loop into a pre-allocated vector, the same loop growing the result with `c()` — the single most common performance bug in student R code — and one vectorised expression.



In [ ]:
## Loop, growing loop and vectorised expression, timed over four problem sizes.
p_activity_loop <- function(v) {                       # pre-allocated loop
  out <- numeric(length(v))
  for (i in seq_along(v)) out[i] <- 9 - log10(v[i])
  out
}
p_activity_grow <- function(v) {                       # the classic bug: the result grows
  out <- numeric(0)
  for (i in seq_along(v)) out <- c(out, 9 - log10(v[i]))
  out
}
p_activity_vec <- function(v) 9 - log10(v)             # one compiled pass over the vector

sizes <- c(1e3, 1e4, 1e5, 1e6)
rows <- list()
for (n in sizes) {
  set.seed(0)
  arr <- runif(n, 0.1, 1e5)                            # concentrations in nM
  mb <- microbenchmark(loop = p_activity_loop(arr), vec = p_activity_vec(arr), times = 3L)
  best <- tapply(mb$time, mb$expr, min) / 1e9          # nanoseconds -> seconds
  grow <- if (n <= 1e4) min(microbenchmark(p_activity_grow(arr), times = 3L)$time) / 1e9 else NA_real_
  rows[[length(rows) + 1]] <- data.frame(n = n, loop_s = best[["loop"]], vector_s = best[["vec"]],
                                         grow_s = grow, speed_up = best[["loop"]] / best[["vec"]])
}
timing <- do.call(rbind, rows)
print(knitr::kable(timing, digits = c(0, 6, 6, 6, 1), format = "simple"))
cat("\nspeed-up of vectorisation at the largest size: about",
    round(timing$speed_up[nrow(timing)]), "times\n")
cat("growing the result instead of pre-allocating costs, at n = 10,000, about",
    round(timing$grow_s[2] / timing$loop_s[2], 1), "times the pre-allocated loop\n")


In [ ]:
## Figure: how the implementations scale with problem size (log-log).
long <- rbind(data.frame(n = timing$n, s = timing$loop_s, impl = "for loop, pre-allocated"),
              data.frame(n = timing$n, s = timing$vector_s, impl = "vectorised expression"),
              data.frame(n = timing$n, s = timing$grow_s, impl = "for loop, growing the result"))
long <- long[!is.na(long$s), ]

p <- ggplot(long, aes(n, s, colour = impl, shape = impl)) +
  geom_line(linewidth = 0.7) + geom_point(size = 2) +
  geom_text(data = timing, aes(x = n, y = vector_s, label = paste0("x", round(speed_up))),
            inherit.aes = FALSE, vjust = 1.9, size = 2.6, colour = PAL[["blue"]]) +
  scale_x_log10() + scale_y_log10() +
  scale_colour_manual(values = unname(PAL[c("orange", "blue", "purple")])) +
  labs(x = "number of concentrations converted", y = "best of 3 runs (s)",
       colour = NULL, shape = NULL, title = "Vectorisation is not a style preference") +
  theme(legend.position = "bottom", legend.text = element_text(size = 7.5))
print(p)



<p class="caption"><b>Figure 2.</b> Converting <i>n</i> concentrations to the logarithmic potency scale by a pre-allocated loop, by a loop that grows its result, and by one vectorised expression, best of three runs, on log–log axes. The loop and the vectorised expression are both straight with slope one — both are linear in <i>n</i> — separated by a roughly constant vertical gap, the per-element interpreter overhead; the annotations give the measured speed-up at each size. The growing loop is steeper than slope one, because every <code>c()</code> allocates a new vector and copies the old one, so the total work is quadratic; it is shown only up to ten thousand elements because a million would not finish in the time budget of this page.</p>

The lesson generalises beyond speed: a vectorised expression is also *shorter*, and shorter code has fewer places for an index error to hide. When you find yourself writing `for (i in 1:nrow(df))`, stop: there is almost always an expression for it, and `vapply()`, `Map()` or `data.table`'s `by=` says a genuinely per-row operation more clearly than a loop.

<div class="box warn"><h4>When a loop is the right answer</h4>
<p>Vectorisation has limits. A loop is correct and appropriate when each iteration is expensive and independent (fitting a model per compound, parsing a file per simulation replica), when the operation is genuinely sequential (a Markov chain, an iterative optimiser), or when the vectorised version would materialise an object that does not fit in memory. The rule is not "never loop"; it is "never loop over elements of a vector to do arithmetic on them, and never grow the result inside the loop". If you must loop, pre-allocate with <code>numeric(n)</code> or <code>vector("list", n)</code> and assign into it.</p></div>

### 3.2 Recycling, and the standardisation idiom

**Recycling** is R's rule for combining vectors of different lengths: the shorter one is repeated until it matches the longer one. When the longer length is an exact multiple of the shorter, R does this *silently*; otherwise it recycles anyway and emits a warning. Combined with the fact that a matrix is a vector with a `dim` attribute, recycling is how column-wise arithmetic is written — and it is also the most common source of silent bugs in R.



In [ ]:
## Recycling: centre and scale a small descriptor matrix, three ways, and check they agree.
X <- matrix(c(320.4, 3.1, 58.2,            # 4 compounds x 3 descriptors (MolWt, logP, TPSA)
              455.9, 4.4, 74.1,
              289.3, 2.2, 49.3,
              512.6, 5.0, 96.5), nrow = 4, byrow = TRUE,
            dimnames = list(NULL, c("MolWt", "MolLogP", "TPSA")))
col_mean <- colMeans(X)
col_sd   <- apply(X, 2, sd)                # note: R's sd() divides by n - 1
cat("X dim", dim(X), "| column means", round(col_mean, 2), "| column sds", round(col_sd, 2), "\n")

Z_sweep  <- sweep(sweep(X, 2, col_mean, "-"), 2, col_sd, "/")   # explicit, dimension-aware
Z_scale  <- scale(X)                                            # the idiom: base R, same result
Z_loop   <- X
for (j in seq_len(ncol(X))) Z_loop[, j] <- (X[, j] - col_mean[j]) / col_sd[j]

cat("\nstandardised matrix (sweep):\n"); print(round(Z_sweep, 3))
cat("\nall three agree:",
    isTRUE(all.equal(Z_sweep, Z_loop)) &&
    isTRUE(all.equal(Z_sweep, matrix(as.numeric(Z_scale), nrow = 4, dimnames = dimnames(X)))), "\n")
cat("column means after standardising:", round(colMeans(Z_sweep), 12), "\n")
cat("column sds   after standardising:", round(apply(Z_sweep, 2, sd), 12), "\n")



The three implementations give identical numbers, and only the second is worth writing in everyday code. `scale()` centres and scales the columns of a matrix in one call; `sweep()` is the explicit form that names the dimension it operates on, and is what you write when the statistic is not the mean or when you must apply a *training* mean to a *test* matrix (Unit 11 — doing that with `scale()` recomputed on the test set is leakage). Note the last two printed lines: after standardisation every column has mean 0 and standard deviation 1 to numerical precision, which is the definition of the z-score used from Unit 11 onwards. One difference from the Python edition: R's `sd()` and `scale()` divide by \( n-1 \), while NumPy's default divides by \( n \), so the same matrix standardised in the two languages differs by a factor of \( \sqrt{n/(n-1)} \) — here about 1.15 on four rows, and negligible on four thousand.

Recycling is also the most common source of silent length bugs, and the bug is worth producing deliberately once so that you recognise its signature: a result whose length is not the length you expected.



In [ ]:
## A failure mode worth producing on purpose: two vectors of different length, silently recycled.
pred <- c(6.1, 7.4, 5.2, 8.0, 6.6, 7.0, 5.9, 8.2)   # 8 predicted pActivity values
obs  <- c(6.0, 7.1, 5.5, 7.8)                        # only 4 observations survived an earlier filter

wrong <- pred - obs          # 8 and 4: an exact multiple, so R recycles with NO warning at all
right <- pred[1:4] - obs     # the matched comparison the analyst intended

cat("lengths:", length(pred), length(obs), "\n")
cat("pred - obs -> length", length(wrong), ":", round(wrong, 2), "\n")
cat("RMSE computed from the recycled vector:", round(sqrt(mean(wrong^2)), 3), "\n")
cat("RMSE computed from the matched vectors:", round(sqrt(mean(right^2)), 3), "\n")

mismatch <- tryCatch(withCallingHandlers(pred - obs[1:3],
                       warning = function(w) { cat("\nnon-multiple lengths DO warn:",
                                                   conditionMessage(w), "\n"); invokeRestart("muffleWarning") }),
                     error = function(e) NULL)
cat("length of the warned result:", length(mismatch), "\n")
cat("the guard that removes the whole family:",
    tryCatch({ stopifnot(length(pred) == length(obs)); "passed" },
             error = function(e) paste("stopifnot() stopped:", conditionMessage(e))), "\n")



Read the two root-mean-square errors. The incorrect one is not a crash and not an absurd number — it is a plausible error metric, computed from eight differences of which four compare a prediction with the wrong observation, and **no warning was printed**, because eight is an exact multiple of four. Only the non-multiple case warns, which means the dangerous case is precisely the quiet one. The defence is mechanical: `stopifnot(length(pred) == length(obs))` at the top of an evaluation function costs one line and removes a whole family of silent errors. In the Python edition the same mistake appears as a shape bug — a result that is suddenly square — and the remedy is the same assertion.

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-run the chunk with <code>obs</code> of length 5 instead of 4 and predict, before you run it, whether R warns and what the length of the result will be. Then wrap the subtraction in a function <code>residuals_checked(pred, obs)</code> that calls <code>stopifnot()</code> first, and confirm that it stops on both the length-4 and the length-5 case. Finally, replace <code>stopifnot()</code> with <code>if (length(pred) != length(obs)) stop("...")</code> carrying an informative message that prints both lengths — this is the version you want in <code>R/</code>, because the message tells a collaborator what went wrong without reading your code.</p></div>

### 3.3 Random numbers that can be reproduced

Anything in this course that uses randomness — a train/test split, a bootstrap resample, the initialisation of a model, a permutation test — must be reproducible, because a reviewer or a colleague has to obtain your numbers. R keeps **one global random stream**, set with `set.seed(k)`; unlike the Python edition, which passes an explicit generator object around, R code must therefore seed immediately before every stochastic step, and the position of `set.seed()` in the document is part of the analysis.



In [ ]:
## Seeds: same seed, same numbers; different seed, different numbers; and the advancing-state trap.
set.seed(0); sample_a <- rnorm(5)
set.seed(0); sample_b <- rnorm(5)
set.seed(1); sample_c <- rnorm(5)
cat("seed 0, draw 1:", round(sample_a, 4), "\n")
cat("seed 0, draw 2:", round(sample_b, 4), " identical:", identical(sample_a, sample_b), "\n")
cat("seed 1, draw 1:", round(sample_c, 4), " identical to seed 0:", identical(sample_a, sample_c), "\n")

set.seed(0); first <- rnorm(5); second <- rnorm(5)   # one seed, two consecutive draws
cat("\nsame seed, two consecutive draws differ:", !identical(first, second), "\n")
cat("generator in use:", RNGkind()[1], "| normal method:", RNGkind()[2], "\n")

paths <- do.call(rbind, lapply(c(0, 1, 2), function(s) {
  set.seed(s); data.frame(step = 1:200, value = cumsum(rnorm(200)), seed = paste("seed", s))
}))
set.seed(0)
again <- data.frame(step = 1:200, value = cumsum(rnorm(200)), seed = "seed 0, recreated later")

p <- ggplot(paths, aes(step, value, colour = seed)) +
  geom_line(data = again, aes(step, value), inherit.aes = FALSE,
            colour = PAL[["blue"]], linewidth = 2.4, alpha = 0.3) +
  geom_line(linewidth = 0.55) +
  geom_hline(yintercept = 0, linetype = "dotted", colour = "grey50") +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green")])) +
  labs(x = "step", y = "cumulative sum of draws", colour = NULL,
       title = "A seed is a promise: the same seed redraws the same path") +
  theme(legend.position = "bottom")
print(p)



<p class="caption"><b>Figure 3.</b> Cumulative sums of 200 standard normal draws from three seeded streams. The thin blue line and the thick pale-blue line coincide exactly — the same seed, set at two different moments in the chunk — while the orange and green paths wander elsewhere. The figure shows the only property of a seed that matters (reproducibility, not "randomness quality") and the warning that comes with it: the paths differ by several units at step 200, so any statistic read off a single seeded run carries that much arbitrary variation. Unit 2 repeats this experiment on a real model; Unit 12 turns it into the discipline of repeated splits.</p>

<div class="box key"><h4>The seeding rule used in this course</h4>
<p>Call <code>set.seed(0)</code> in the setup chunk <em>and</em> immediately before every split, resample or simulation, because R's stream is global and advances with every draw — a chunk that is re-run out of order otherwise produces different numbers from a document that was knitted linearly. Never seed inside a loop that is supposed to produce independent replicates (that makes every replicate identical), never rely on the state left by an earlier chunk, and when you report a number that depends on a split, report it over several seeds rather than the one that looked best. If a function you call samples internally, check its documentation for a seed argument; if it has none, <code>set.seed()</code> before calling it.</p></div>

## 4. Tables: data frames, tibbles and data tables

A matrix is a rectangle of values of one type. Real data arrives as a rectangle of numbers *and* text, with named columns, missing entries and a meaningful row identity — in other words, as a table. R supplies three closely related objects for that. The **`data.frame`** is base R's table: a list of equal-length vectors, one per column. A **`tibble`** is the tidyverse's data frame with sharper defaults — it never converts strings to factors, never does partial name matching, and prints a few rows with their types instead of flooding the console. A **`data.table`** is a data frame extended with a concise query grammar and modify-in-place semantics, which makes it the fastest of the three on large tables.

The part newcomers underestimate is what R does *not* do: unlike a pandas `Series`, a named R vector does **not** align on names in arithmetic.



In [ ]:
## Named vectors do NOT align by name in R - they recycle by position. A real trap.
ic50_nm <- c(CHEMBL1 = 25, CHEMBL2 = 2500, CHEMBL3 = 1)
weight  <- c(CHEMBL2 = 455.9, CHEMBL3 = 289.3, CHEMBL9 = 410.0)
cat("a named vector carries names:\n"); print(ic50_nm)
cat("\nic50_nm / weight  -- aligned by POSITION, names taken from the first argument:\n")
print(round(ic50_nm / weight, 4))
cat("\nCHEMBL1 was divided by CHEMBL2's weight, and nothing warned.\n")

## The explicit, correct way: a join on the identifier.
joined <- merge(data.frame(id = names(ic50_nm), IC50_nM = as.numeric(ic50_nm)),
                data.frame(id = names(weight),  MolWt  = as.numeric(weight)),
                by = "id", all = TRUE)
cat("\nmerge(all = TRUE) keeps the UNION of the identifiers and inserts NA:\n")
print(joined)
cat("\nrows:", nrow(joined), "| complete rows:", sum(complete.cases(joined)), "\n")



That output is the single most important difference between R and pandas for a reader coming from the Python edition. pandas aligns two `Series` on their index automatically, taking the union of the labels and inserting missing values; R recycles by position and keeps the names of the first argument, so `CHEMBL1` was silently divided by `CHEMBL2`'s molecular weight. **In R, alignment is something you do, not something that happens.** The explicit form is a join: `merge()` in base R, `dplyr::full_join()` in the tidyverse, or `X[Y, on = "id"]` in `data.table`. The habit that prevents the surprise is to check `nrow()` before and after any operation that combines tables, and to count missing values afterwards.

### 4.1 Loading and inspecting a real table

Now the real file. We read a **200-row extract** of the course's raw ChEMBL snapshot — `nrows = 200` — because this unit practises loading and inspecting, not chemistry. The full 5,089-row table is the subject of Units 3 to 6.



In [ ]:
## First contact with real data: a 200-row extract, read through the course loader.
ex <- load_data("rock1_chembl_raw.csv", nrows = 200)
cat("dim:", dim(ex), "(rows, columns)\n")
cat("class:", paste(class(ex), collapse = " / "), "\n\n")
cat("columns:", paste(names(ex), collapse = ", "), "\n\n")
cat(sprintf("memory held: %.3f MB\n", as.numeric(object.size(ex)) / 1e6))
print(knitr::kable(head(ex[, .(molecule_chembl_id, standard_type, standard_relation,
                               standard_value, standard_units, assay_type)], 8), format = "simple"))



<p class="caption"><b>Table (Section 4.1).</b> The first eight records of the extract, restricted to six of the twenty-five columns. Each row is one measurement: a compound identifier, the kind of measurement (<code>standard_type</code>), a relation symbol, a value in nanomolar, and the type of assay. Already visible is the feature that makes this table a teaching dataset rather than a clean one: the relation column is not always <code>=</code>.</p>

Four calls answer the first four questions you should ask of any table, and you should ask them in this order, every time, before computing anything: `dim()` (how big?), `sapply(df, class)` or `str()` (what is each column, really?), `head()` (what does a row look like?), and `summary()` together with `colSums(is.na(df))` (what is the range, and what is missing?).



In [ ]:
## The four questions: storage types, value ranges, missingness, and category counts.
work_cols <- c("standard_value", "pchembl_value", "max_phase", "standard_type",
               "standard_relation", "potential_duplicate")
cat("class of the six working columns:\n")
print(sapply(ex[, ..work_cols], function(x) class(x)[1]))

na_count <- sort(colSums(is.na(ex)), decreasing = TRUE)
cat("\nmissing values per column (top 8):\n"); print(na_count[1:8])
cat("\nnote: fread() reads empty strings as \"\", not NA, in character columns -",
    "count both:\n")
blankish <- sapply(ex, function(x) if (is.character(x)) sum(x == "" | is.na(x)) else sum(is.na(x)))
print(sort(blankish, decreasing = TRUE)[1:5])

cat("\nmeasurement types :", paste(names(table(ex$standard_type)),
                                   table(ex$standard_type), sep = "=", collapse = "  "), "\n")
cat("relation symbols  :", paste(names(table(ex$standard_relation, useNA = "ifany")),
                                 table(ex$standard_relation, useNA = "ifany"),
                                 sep = "=", collapse = "  "), "\n")
cat("reported units    :", paste(names(table(ex$units, useNA = "ifany")),
                                 table(ex$units, useNA = "ifany"), sep = "=", collapse = "  "), "\n")
cat("\nnumeric summary of standard_value (nM):\n")
print(round(summary(ex$standard_value), 2))



Read that output as a damage report, because it is one. The measurements are mostly IC50 with a minority of Ki, so two different physical quantities share one column. The relation symbols are not all `=`: a substantial fraction are `>`, meaning "inactive up to the highest concentration tested, so the true value lies above this number" — these are **censored** measurements, and treating them as exact numbers is one of the standard errors of the field. Several rows carry an empty relation string, the reported units mix nanomolar with blanks, and `standard_value` spans several orders of magnitude, which is why Section 5 converts it to a logarithmic scale before anything else happens to it. None of this is cleaned here; the point is that you can *see* it within four commands of opening the file.

Note the second block of that output, which has no counterpart in the Python edition: `fread()` leaves an empty field in a character column as the empty string, while `read.csv()` would give `NA` and, in older R, a factor level. **`colSums(is.na())` therefore undercounts missingness in character columns read by `fread`**, and you must count `x == "" | is.na(x)` instead. This is the first of several places where the two R readers disagree, and the reason the setup chunk prints which one it used.



In [ ]:
## Name-based versus position-based indexing, and the partial-matching trap of $.
sub <- as.data.frame(ex)
rownames(sub) <- sub$activity_id          # the ChEMBL activity id becomes the row label
first_label <- rownames(sub)[1]

cat("by NAME     :", first_label, "->", sub[first_label, "standard_type"],
    sub[first_label, "standard_value"], "\n")
cat("by POSITION : row 1          ->", sub[1, "standard_type"], sub[1, "standard_value"], "\n")

## A logical mask selects rows by condition: the R equivalent of a WHERE clause (Unit 3).
mask <- ex$standard_type == "IC50" & ex$standard_relation == "=" & !is.na(ex$standard_value)
exact_ic50 <- ex[mask, .(molecule_chembl_id, standard_value, standard_units)]
cat(sprintf("\nexact IC50 rows: %d of %d (%.0f%%)\n",
            nrow(exact_ic50), nrow(ex), 100 * nrow(exact_ic50) / nrow(ex)))
cat(sprintf("median of those values: %.1f nM\n", median(exact_ic50$standard_value)))

## The trap: $ on a data.frame matches names PARTIALLY; on a data.table and a tibble it does not.
cat("\ndata.frame partial matching: sub$standard_val returns",
    if (is.null(sub$standard_val)) "NULL" else paste("the full column, length", length(sub$standard_val)), "\n")
cat("data.table                 : ex$standard_val returns",
    if (is.null(ex$standard_val)) "NULL" else "a column", "\n")
print(knitr::kable(head(exact_ic50, 5), format = "simple"))



<p class="caption"><b>Table (Section 4.1).</b> The first five exactly-measured IC50 records, selected with a logical mask. The counts printed above the table state how many of the 200 records survive the three conditions (an IC50, an <code>=</code> relation, a non-missing value) — the first quantitative appearance in this course of the gap between "records downloaded" and "records usable".</p>

Index by name when the row identity is meaningful and by position when it is not, and never mix them. Two traps are visible above. The first is **partial matching**: `sub$standard_val` returned the whole `standard_value` column from the `data.frame`, because base R's `$` matches a prefix, while the same expression on the `data.table` returned `NULL` — a typo is a silent success in one object and an obvious failure in the other. The second is that base R's `[` **drops dimensions** when a selection leaves one column: `sub[, "standard_value"]` is a vector, not a one-column data frame, which breaks code downstream that expected a table. Write `drop = FALSE`, or use a `data.table` or `tibble`, which never drop.

<div class="box warn"><h4>Three R-specific reading traps</h4>
<p><strong>Strings and factors.</strong> <code>read.csv()</code> in R 4.x no longer converts strings to factors by default, but code and tutorials written before R 4.0 assumed it did, and <code>read.table()</code> wrappers in older packages still do. <code>fread()</code> and <code>readr::read_csv()</code> never do. A factor read as a factor and then passed to <code>as.numeric()</code> returns the <em>level codes</em>, not the numbers — the single most destructive silent error in legacy R scripts. Always check with <code>str()</code>.</p>
<p><strong>Empty strings.</strong> As shown above, <code>fread()</code> gives <code>""</code> where <code>read.csv()</code> gives <code>NA</code>. Pass <code>na.strings = c("", "NA")</code> to <code>fread()</code> when an empty field means missing.</p>
<p><strong>Modify-in-place.</strong> <code>data.table</code>'s <code>:=</code> modifies the table without copying it, so a function that receives a <code>data.table</code> and uses <code>:=</code> changes its caller's object — the one place where R's copy-on-modify guarantee does not hold. Use <code>copy()</code> when you intend a copy.</p></div>

### 4.2 Storage types, factors and memory

A column's storage type determines how much memory it occupies, which operations are available on it, and how missing values behave. Three choices matter in practice: integer versus double (`1L` versus `1.0`, a factor of two in memory), character versus **factor** (a text column with few distinct values is cheaper and faster as a factor, which stores each level once plus one integer code per row), and logical versus integer for flags.



In [ ]:
## How much memory each column costs, and what a factor buys on a low-cardinality column.
cols <- c("standard_type", "standard_units", "assay_type", "assay_description",
          "canonical_smiles", "standard_value")
before <- sapply(cols, function(c) as.numeric(object.size(ex[[c]])))
converted <- lapply(cols, function(c)
  if (c %in% c("standard_type", "standard_units", "assay_type")) factor(ex[[c]]) else ex[[c]])
after <- sapply(converted, function(x) as.numeric(object.size(x)))
mem <- data.frame(column = cols, bytes_before = before, bytes_after = after,
                  n_unique = sapply(cols, function(c) length(unique(ex[[c]]))),
                  saving_pct = round(100 * (1 - after / before), 1))
print(knitr::kable(mem, row.names = FALSE, format = "simple"))
cat(sprintf("\ntotal for these six columns: %.1f kB -> %.1f kB\n", sum(before) / 1e3, sum(after) / 1e3))

m <- rbind(data.frame(column = cols, kB = before / 1e3, state = "as read (character/double)"),
           data.frame(column = cols, kB = after / 1e3, state = "low-cardinality as factor"))
m$column <- factor(m$column, levels = rev(cols))
p <- ggplot(m, aes(kB, column, fill = state)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.7) +
  scale_fill_manual(values = unname(PAL[c("orange", "blue")])) +
  labs(x = "memory held by the column (kB)", y = NULL, fill = NULL,
       title = "Storage type is a memory decision") +
  theme(legend.position = "bottom", axis.text.y = element_text(size = 7.5))
print(p)



<p class="caption"><b>Figure 4.</b> Memory held by six columns of the extract, before (orange) and after (blue) converting the three low-cardinality text columns to factors. The converted columns shrink substantially — a factor stores each distinct string once plus a small integer code per row — while <code>canonical_smiles</code> (almost every value distinct) and the numeric column are untouched, correctly: converting a high-cardinality column to a factor <em>costs</em> memory. The <code>n_unique</code> column of the printed table is the deciding quantity. Note that R's global string cache already shares repeated character values, so the measured saving here is smaller than the equivalent pandas <code>category</code> conversion reports in the Python edition — the two languages solve the same problem in different places.</p>

On 200 rows the saving is irrelevant; on a ten-million-row screening output (Unit 19) it decides whether the table loads at all.



In [ ]:
## Where the missing values are: a map of the extract (records x columns).
miss <- as.data.frame(lapply(ex, function(x) if (is.character(x)) (x == "" | is.na(x)) else is.na(x)))
pct <- round(100 * colMeans(miss))
long <- data.frame(record = rep(seq_len(nrow(miss)), times = ncol(miss)),
                   column = factor(rep(names(miss), each = nrow(miss)), levels = rev(names(miss))),
                   missing = as.vector(as.matrix(miss)))
lab <- data.frame(column = factor(names(pct), levels = rev(names(miss))),
                  label = ifelse(pct > 0, paste0(pct, "%"), ""))

p <- ggplot(long, aes(record, column, fill = missing)) +
  geom_raster() +
  geom_text(data = lab, aes(x = nrow(miss) + 14, y = column, label = label),
            inherit.aes = FALSE, size = 2.1, hjust = 0) +
  scale_fill_manual(values = c(`FALSE` = "grey92", `TRUE` = "grey20"), guide = "none") +
  scale_x_continuous(limits = c(0, nrow(miss) + 34), expand = c(0, 0)) +
  labs(x = "record (row) in the 200-row extract", y = NULL,
       title = "Missingness map: dark = missing or empty") +
  theme(axis.text.y = element_text(size = 5.6), panel.grid = element_blank())
print(p)
cat("columns with no missing or empty value:", sum(colSums(miss) == 0), "of", ncol(ex), "\n")



<p class="caption"><b>Figure 5.</b> A missingness map: every column of the table is a row of the image, every record a column of pixels, and a dark pixel marks a missing or empty entry. The annotation at the right gives the missing fraction per column. The annotation columns (clinical phase, preferred name, validity and activity comments) are empty for the great majority of records, while identifiers and standardised measurements are nearly complete. Patterns matter more than totals — a column empty in a contiguous block usually marks a group of records from one source, the structure Units 5 and 6 exploit.</p>

This figure takes six lines and should be the second thing you produce for any new table, right after `dim()`.



In [ ]:
## Split-apply-combine, in dplyr and in data.table: the operation you will use in every project.
grp_dplyr <- ex |>
  group_by(standard_type) |>
  summarise(n = n(),
            n_exact = sum(standard_relation == "=", na.rm = TRUE),
            median_nM = median(standard_value, na.rm = TRUE),
            min_nM = min(standard_value, na.rm = TRUE),
            max_nM = max(standard_value, na.rm = TRUE),
            n_assays = dplyr::n_distinct(assay_chembl_id), .groups = "drop") |>
  arrange(desc(n))

grp_dt <- ex[, .(n = .N,
                 n_exact = sum(standard_relation == "=", na.rm = TRUE),
                 median_nM = median(standard_value, na.rm = TRUE),
                 min_nM = min(standard_value, na.rm = TRUE),
                 max_nM = max(standard_value, na.rm = TRUE),
                 n_assays = uniqueN(assay_chembl_id)),
             by = standard_type][order(-n)]

print(knitr::kable(grp_dplyr, digits = 2, format = "simple"))
cat("\ndata.table gives the same table:",
    isTRUE(all.equal(as.data.frame(grp_dplyr), as.data.frame(grp_dt), check.attributes = FALSE)), "\n")
cat("total records:", sum(grp_dplyr$n), "| exactly measured:", sum(grp_dplyr$n_exact), "\n")



<p class="caption"><b>Table (Section 4.2).</b> The extract summarised by measurement type: how many records, how many of them exact, the median and extreme values in nanomolar, and how many distinct assays contributed. The spread between <code>min_nM</code> and <code>max_nM</code> within a single measurement type covers several orders of magnitude — the quantitative justification for the logarithmic scale of Section 5.</p>

`group_by() |> summarise()` is the **split–apply–combine** pattern: split the rows into groups by a key, apply a reduction to each group, combine the results into a new table<sup class="c"><a href="#r3">[3]</a></sup>. It is the same operation as SQL's `GROUP BY` (Unit 3), and the `data.table` form `DT[, .(...), by = key]` computes it identically — the printed check confirms that the two agree on every value, which is worth doing once so that you trust both. The pattern will appear in every remaining unit of this course.

<div class="exercise"><h4>Exercise 2</h4>
<p>Group the extract by <code>assay_type</code> and add a column counting distinct source documents (<code>document_chembl_id</code>). Then group by both columns at once — <code>group_by(assay_type, standard_type)</code>, or <code>by = .(assay_type, standard_type)</code> in <code>data.table</code> — and say in one sentence what a row of the result means. Finally, convert both columns to factors with <code>factor(..., levels = c(...))</code> including a level that does not occur in the data, and compare what <code>dplyr::count()</code> and <code>table()</code> report for the empty level: one drops it and one keeps it with a count of zero. Which behaviour do you want when the table goes into a paper?</p></div>

## 5. A worked example by hand: from nanomolar to pActivity

Every unit of this course traces at least one calculation by hand and then confirms it by code, because a number produced with a pencil is a number you can debug. Here it is the conversion that every bioactivity table in the group passes through. A potency is reported as a concentration \( c \), usually in nanomolar (nM, \( 10^{-9} \) mol/L); concentrations are multiplicative in nature — ten times more potent is a *factor*, not a difference — so the field works with the negative base-10 logarithm of the molar concentration:

\[ \mathrm{pActivity} \;=\; -\log_{10}\!\big(c\,[\mathrm{M}]\big) \;=\; -\log_{10}\!\big(c\,[\mathrm{nM}] \times 10^{-9}\big) \;=\; 9 - \log_{10}\!\big(c\,[\mathrm{nM}]\big). \]

The last form is the one to memorise: **subtract the log of the nanomolar value from 9**. A larger pActivity means a more potent compound, and one unit of pActivity is one order of magnitude in concentration. Tracing four values by hand:

| \( c \) (nM) | \( \log_{10} c \) | \( 9 - \log_{10} c \) | pActivity |
|---|---|---|---|
| 1 | \( \log_{10} 1 = 0 \) | \( 9 - 0 \) | **9.00** |
| 25 | \( \log_{10} 25 = \log_{10}(100/4) = 2 - 0.602 = 1.398 \) | \( 9 - 1.398 \) | **7.60** |
| 2 500 | \( \log_{10} 2500 = \log_{10}(10^4/4) = 4 - 0.602 = 3.398 \) | \( 9 - 3.398 \) | **5.60** |
| 100 000 | \( \log_{10} 10^5 = 5 \) | \( 9 - 5 \) | **4.00** |

The middle two rows use the one logarithm worth knowing by heart, \( \log_{10} 4 \approx 0.602 \) (because \( \log_{10} 2 \approx 0.301 \)). Note the structure the table reveals: a hundred-fold change in concentration (25 nM to 2 500 nM) is exactly two units of pActivity, and the ratio of the two concentrations, not their difference, is what the scale preserves. The code below confirms all four values and then applies the same expression to the extract.



In [ ]:
## Confirming the hand calculation, then applying it to the real column.
hand <- data.frame(c_nM = c(1, 25, 2500, 1e5), by_hand = c(9.00, 7.60, 5.60, 4.00))
hand$by_code <- round(9 - log10(hand$c_nM), 2)
hand$agree <- abs(hand$by_hand - hand$by_code) < 0.005
print(knitr::kable(hand, format = "simple"))
cat("\nall four hand values confirmed:", all(hand$agree), "\n")

## Apply to the extract: exact measurements reported in nM, any measurement type.
work <- ex[standard_relation == "=" & !is.na(standard_value) & standard_units == "nM"]
work[, pActivity := 9 - log10(standard_value)]
cat("\nexact nM records in the extract:", nrow(work), "\n")
print(round(summary(work$pActivity), 2))
has_pchembl <- !is.na(work$pchembl_value)
cat(sprintf("\nChEMBL's own pchembl_value agrees with ours to 2 decimals in %d of %d rows that have it\n",
            sum(abs(work$pActivity[has_pchembl] - work$pchembl_value[has_pchembl]) < 0.005),
            sum(has_pchembl)))



The four hand-computed values are confirmed to two decimals, and our conversion agrees with ChEMBL's own `pchembl_value` on all but a handful of the records that carry one; the printed counts give both numbers, and the exceptions are records whose reported and standardised values differ. That is a *cross-check*, not a formality: it confirms that we read the right column, in the right units, with the right sign. Whenever a dataset ships a derived column, recompute it and compare — the descriptor columns of the curated table used from Unit 7 onwards were produced the same way, by a documented toolchain<sup class="c"><a href="#r30">[30]</a></sup>, and the same recomputation applies to them.

<div class="box key"><h4>Why the logarithm is not optional</h4>
<p>On the raw nanomolar scale, one compound at 0.5 nM and another at 1 nM differ by 0.5 units, while two inactive compounds at 50 000 and 100 000 nM differ by 50 000 — so an unweighted mean, a least-squares fit or a Euclidean distance computed on raw concentrations is dominated entirely by the weakest compounds. On the pActivity scale both pairs differ by about 0.30 units, which is the chemically meaningful statement "a factor of two". Every statistical method in Blocks C and D assumes the modelling target is on a scale where distances mean something; for potencies, that scale is logarithmic.</p></div>

## 6. Notebook and R-Markdown discipline

An interactive R session has one property no script has: the order in which you ran the pieces is invisible in the document. A notebook or an R-Markdown file open in an editor shows chunks from top to bottom, and nothing records that chunk 12 was run before chunk 7, that chunk 9 was edited after it was executed, or that a variable was created in a chunk you later deleted. This is **hidden state**, and it is the reason a large fraction of published notebooks cannot be re-executed: a study of 1.16 million Jupyter notebooks on GitHub found that only a minority could be run top-to-bottom without error, and of those a smaller fraction reproduced the stored results<sup class="c"><a href="#r19">[19]</a></sup>. The diagnosis is not carelessness; it is that the medium does not warn you.

R-Markdown removes part of the problem by construction, and this is the main practical reason the group writes analyses as `.Rmd` rather than as a live session: `rmarkdown::render()` knits in a **fresh R session** by default, so a document that knits is a document that runs from nothing<sup class="c"><a href="#r6">[6]</a></sup><sup class="c"><a href="#r7">[7]</a></sup>. That is exactly the "restart and run all" discipline, enforced by the tool instead of by you. It does not remove the problem in the editor: the chunks you run interactively while writing share a global environment that the knit does not see, which is how a document comes to depend on an object that no chunk creates.

The following chunk makes the failure concrete by executing the same three "chunks" in two different orders, each in its own environment.



In [ ]:
## Hidden state, demonstrated: the same three chunks, two execution orders, two different answers.
cell_A <- "threshold <- 1000"                                     # the analyst sets a cut-off in nM
cell_B <- "selected <- work[standard_value <= threshold]"         # ... and selects on it
cell_C <- "threshold <- 50"                                       # later, while exploring, it changes

run_order <- function(order) {
  env <- new.env(parent = globalenv())
  assign("work", work, envir = env)
  threshold_used <- NULL
  for (src in order) {
    eval(parse(text = src), envir = env)
    if (exists("selected", envir = env, inherits = FALSE) && is.null(threshold_used))
      threshold_used <- get("threshold", envir = env)
  }
  list(n = nrow(get("selected", envir = env)), thr = threshold_used)
}

top_to_bottom <- run_order(list(cell_A, cell_B, cell_C))   # what the SAVED document shows
as_explored   <- run_order(list(cell_A, cell_C, cell_B))   # what the analyst actually did
cat(sprintf("order A -> B -> C  (knit from scratch) : threshold = %7.1f nM, rows selected = %d\n",
            top_to_bottom$thr, top_to_bottom$n))
cat(sprintf("order A -> C -> B  (as it was explored): threshold = %7.1f nM, rows selected = %d\n",
            as_explored$thr, as_explored$n))
cat("\nthe document looks identical in both cases; the result differs by",
    abs(top_to_bottom$n - as_explored$n), "rows\n")

d <- data.frame(order = c("A -> B -> C\n(knit from scratch)", "A -> C -> B\n(as explored)"),
                n = c(top_to_bottom$n, as_explored$n))
p <- ggplot(d, aes(order, n, fill = order)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = n), vjust = -0.4, fontface = "bold", size = 3.2) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.12))) +
  labs(x = NULL, y = "records selected",
       title = "Same document, same chunks, different execution order")
print(p)



<p class="caption"><b>Figure 6.</b> Records selected by the identical three chunks, executed top-to-bottom (blue) and in the order an analyst actually explored them (orange). The saved document looks the same in both cases, because it stores chunks and outputs but not the execution history; the difference between the bars is the hidden state. Changing a threshold in a chunk above the one that uses it, while working downwards in the editor, is the most common way to produce it.</p>

The discipline that removes the problem is mechanical and non-negotiable in this group.

1. **Knit from a clean session before saving, and before showing a result to anyone.** If the document fails to knit, the document is wrong. From the shell: `Rscript -e 'rmarkdown::render("analysis.Rmd")'`.
2. **One concept per chunk, and no chunk that depends on a chunk below it.** A document should read as an argument, top to bottom.
3. **No hidden deletions.** If you delete a chunk that defined something, restart.
4. **Every chunk's output is discussed.** An output nobody reads is an output nobody checks.
5. **Parameters at the top.** Thresholds, paths and seeds live in one chunk near the beginning, never buried in the middle; `params:` in the YAML header is the formal version of this.
6. **Name your chunks.** A named chunk (`` ```{r load-extract} ``) gives you a navigable document, a readable error message when the knit fails, and reliable caching; unnamed chunks produce errors that say "chunk 17".

These rules are the consensus of the "ten simple rules" literature for computational notebooks<sup class="c"><a href="#r20">[20]</a></sup> and of the good-enough-practices guidance for scientific computing generally<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r21">[21]</a></sup>; the argument that the analysis document and the manuscript should be one artefact is older than either, and was made for R specifically<sup class="c"><a href="#r8">[8]</a></sup>. Jupyter notebooks remain the lingua franca for sharing executable analyses<sup class="c"><a href="#r13">[13]</a></sup><sup class="c"><a href="#r14">[14]</a></sup>, which is why every unit of this course ships one with an R kernel alongside the knitted page.

### 6.1 When a document becomes a script

An R-Markdown document stops being the right container at a predictable moment: when a piece of code is used **twice**. Three symptoms say the moment has arrived — you have copy-pasted a function from one document into another; a chunk has grown past roughly fifty lines; or you want to test a piece of logic in isolation. The remedy is to move the function into `R/`, `source()` it back, and leave the document to narrate the analysis.



In [ ]:
## Promoting a chunk to a script in R/, and sourcing it back.
project <- normalizePath(file.path(tempdir(), "demo_project"), mustWork = FALSE)
dir.create(file.path(project, "R"), recursive = TRUE, showWarnings = FALSE)

module_source <- c(
  "# Activity-scale helpers for the ROCK1 project (promoted from the U1 document).",
  "nm_to_pactivity <- function(value_nm) {",
  "  value_nm <- as.numeric(value_nm)",
  "  if (any(value_nm <= 0, na.rm = TRUE))",
  "    stop('concentrations must be strictly positive')",
  "  9 - log10(value_nm)",
  "}",
  "",
  "pactivity_to_nm <- function(p) 10^(9 - as.numeric(p))")
writeLines(module_source, file.path(project, "R", "activity.R"))

helpers <- new.env()
sys.source(file.path(project, "R", "activity.R"), envir = helpers)   # source into its own environment
cat("module file:", file.path("R", "activity.R"), "|",
    length(module_source), "lines\n")
cat("round trip 25 nM -> pActivity -> nM:",
    round(helpers$nm_to_pactivity(25), 4), "->",
    round(helpers$pactivity_to_nm(helpers$nm_to_pactivity(25)), 4), "\n")
cat("agrees with the document's own expression:",
    isTRUE(all.equal(helpers$nm_to_pactivity(work$standard_value), work$pActivity)), "\n")
cat("guard works:", tryCatch(helpers$nm_to_pactivity(0),
                            error = function(e) conditionMessage(e)), "\n")



Three things happened in that chunk and all three are the point. A function was written to a file in `R/`, with a comment and a guard against the invalid input (a concentration of zero, whose logarithm does not exist) that the document version silently allowed; it was sourced back into its own environment — `sys.source(..., envir = )` rather than bare `source()`, so the helpers do not collide with anything in the global environment — and used as any library function; and its output was checked against the document's own expression on the real column, which printed `TRUE`, so the promotion provably did not change the analysis. Unit 24 replaces that manual check with a `testthat` suite that runs on every commit.

<div class="box key"><h4>The rule of two</h4>
<p>Code used once stays in the document. Code used twice moves to <code>R/</code>. Code used by two people gets a test. Code used by two projects gets a package with a version number.</p></div>

## 7. The group's project template

Every project in this laboratory uses the same folder layout. The layout is not a matter of taste: it encodes three rules that make a project reproducible and auditable. Raw data is never modified. Derived data is always regenerable. Code that produces a result is stored next to the result it produces.



In [ ]:
## Create the group's project template from scratch, with a README and a renv placeholder.
template_dirs <- c("data/raw", "data/processed", "data/external",
                   "analysis", "R", "results", "figures", "docs")
for (d in template_dirs) dir.create(file.path(project, d), recursive = TRUE, showWarnings = FALSE)

writeLines(c(
  "# ROCK1 activity project", "",
  "One-paragraph statement of the question, the data and the main result.", "",
  "## Layout",
  "- `data/raw/` immutable downloads (never edited; hashed in `docs/manifest.json`)",
  "- `data/processed/` tables regenerated by `R/` from `data/raw/`",
  "- `analysis/` numbered R-Markdown narratives",
  "- `R/` sourceable functions and scripts",
  "- `results/`, `figures/` outputs, each produced by a named script", "",
  "## Reproduce", "```",
  "R -q -e 'renv::restore()'", "Rscript R/build_dataset.R",
  "Rscript -e 'rmarkdown::render(\"analysis/01_explore.Rmd\")'", "```"),
  file.path(project, "README.md"))

writeLines(c("# renv.lock is written by renv::snapshot() and committed; the library is not.",
             "# Rebuild on another machine with renv::restore().",
             "Project: rock1", "R version: 4.5.3"),
           file.path(project, "renv.lock.README"))

writeLines(c("data/raw/*", "data/processed/*", "!**/.gitkeep", ".Rproj.user", ".Rhistory",
             ".RData", "renv/library/", "*_cache/", "*_files/"),
           file.path(project, ".gitignore"))
for (d in c("data/raw", "data/processed", "data/external", "results", "figures"))
  file.create(file.path(project, d, ".gitkeep"))

listing <- sort(sub(paste0(project, "/"), "", list.files(project, recursive = TRUE, all.files = TRUE,
                                                         full.names = TRUE, no.. = TRUE), fixed = TRUE))
cat(sprintf("created %d directories and %d files under %s/\n",
            length(template_dirs), length(listing), basename(project)))
cat(paste("   ", listing), sep = "\n")



Note the `.gitignore`: the contents of `data/raw/` and `data/processed/` are *not* committed, but the empty `.gitkeep` files are, so the folders exist for anyone who clones the project. Data goes to a repository with a checksum and a DOI (Unit 2), not into git. The R-specific entries matter too: `.RData` and `.Rhistory` are the saved workspace and command history an interactive session writes on exit, and committing them is how hidden state travels to a collaborator; `renv/library/` is rebuilt from the lock file rather than shared; `*_cache/` and `*_files/` are knitr's intermediate directories.



In [ ]:
## The template as a diagram, with the rule each folder enforces.
items <- data.frame(
  name = c("project/", "data/raw/", "data/processed/", "data/external/", "analysis/", "R/",
           "results/", "figures/", "docs/", "renv.lock", "README.md"),
  depth = c(0, rep(1, 10)),
  y = c(0, 0.9, 1.8, 2.7, 3.6, 4.5, 5.4, 6.3, 7.2, 8.1, 9.0),
  fill = c("#f0f0f0", "#f6dcdc", "#f7e7d6", "#f7e7d6", "#dce8f5", "#dce8f5",
           "#dcefdc", "#dcefdc", "#ece7f6", "#ffffff", "#ffffff"),
  note = c("", "immutable; hashed; never edited by code",
           "regenerable from raw by R/ in one command",
           "third-party tables you did not produce",
           "numbered R-Markdown narratives; knit from a clean session",
           "sourceable functions, tested; the rule of two",
           "tables for the paper, each from a named script",
           "figures for the paper, each from a named script",
           "manifest, data dictionary, decisions log",
           "the exact package versions (Unit 2)",
           "question, layout, reproduce-in-three-commands"),
  stringsAsFactors = FALSE)

p <- ggplot(items) +
  geom_tile(aes(x = 1.5 + 0.55 * depth, y = -y, fill = I(fill)), width = 2.6, height = 0.62,
            colour = "grey35", linewidth = 0.4) +
  geom_text(aes(x = 0.3 + 0.55 * depth, y = -y, label = name), hjust = 0, size = 2.8,
            family = "mono", fontface = ifelse(items$depth == 0, "bold", "plain")) +
  geom_text(aes(x = 3.0, y = -y, label = note), hjust = 0, size = 2.5, colour = "grey25") +
  scale_x_continuous(limits = c(0.1, 9.4)) +
  labs(title = "The group's project template, and the rule each folder enforces") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 7.</b> The project template created by the previous chunk, annotated with the rule each folder enforces. Red marks the only folder whose contents are never written by your code; orange marks folders that can be deleted and rebuilt at any time; blue marks code; green the outputs that go into the paper; purple the documentation that makes the whole thing auditable. If you can delete <code>data/processed/</code>, <code>results/</code> and <code>figures/</code>, run one command and get them all back, the project is reproducible — and that single test is the one a reviewer effectively applies.</p>

<div class="box bridge"><h4>Bridge — where this template is used across the group's courses</h4>
<p><a href="../U2_Reproducibility/index.html">Unit 2</a> puts this layout under version control and adds the data manifest; <a href="../U3_Getting_Data/index.html">Unit 3</a> fills <code>data/raw/</code> from APIs and databases. The later units of this R edition are built in the same sequence as the <a href="../../Python/index.html">Python edition</a>, whose counterparts of this unit's successors already exist: <a href="../U4_Tidy_Data/index.html">tidy data</a> produces <code>data/processed/</code>, <a href="../U24_Software_Engineering/index.html">software engineering for scientists</a> turns <code>R/</code> into a tested, installable package, and <a href="../U27_Capstone/index.html">the capstone</a> is carried out on exactly this layout. The cheminformatics elective uses the same skeleton for molecular work — see its units on <a href="../../../Cheminformatics/U6_Databases/index.html">databases</a> and <a href="../../../Cheminformatics/U7_Data_Curation/index.html">curation</a> — while BGM324 shows what the structural tools in <a href="../../../BGM324/03_Molecular_Docking/index.html">molecular docking</a> and <a href="../../../BGM324/05_Molecular_Dynamics/index.html">molecular dynamics</a> deposit into <code>data/raw/</code>, and the R edition of the AI course BGM411 (<a href="../../../BGM411/R/index.html">study material</a>) uses the same <code>R/</code> discipline for its model code.</p></div>

## 8. Working with an AI coding assistant

AI coding assistants are part of the working environment, and pretending otherwise would make this unit obsolete on the day it was written. They are useful for the kind of code this course produces: a `ggplot2` idiom you have forgotten, a `tidyr` reshape, a roxygen comment, a first draft of a function whose logic you already know. They are also, measurably, a source of defects — an audit of assistant-generated code found that a substantial share of security-relevant completions contained vulnerabilities<sup class="c"><a href="#r22">[22]</a></sup>, the underlying models pass functional-correctness benchmarks only partially<sup class="c"><a href="#r23">[23]</a></sup>, and in scientific work the characteristic failure is not a crash but a *plausible wrong number*<sup class="c"><a href="#r24">[24]</a></sup>. Scientific publishing has converged on a simple position: the tool is not an author and the human is accountable for every line<sup class="c"><a href="#r25">[25]</a></sup>.

This group's policy therefore has five steps, in order, and the middle three are not optional.

1. **Draft.** Ask the assistant for a first version, with the context it needs (the column types, the lengths, the units).
2. **Read.** Read every line. If you cannot say what a line does, delete it or learn it — there is no third option.
3. **Test.** Write the check yourself, on a case whose answer you know independently (by hand, from a published value, or from a degenerate case).
4. **Explain.** Be able to explain the code to your supervisor without the assistant open. If you cannot, you do not own it.
5. **Attribute.** Record in the project `README` that an assistant was used and for what. This is the same courtesy as citing a package.

The next chunk shows why step 3 is the load-bearing one, using a draft that is wrong in the way assistants are characteristically wrong: fluent, well-formatted, documented, and off by a constant.



In [ ]:
## Step 3 in action: a plausible 'assistant draft' and the four-line test that rejects it.
pactivity_draft <- function(value_nm) {
  # Convert a concentration to pActivity. (A plausible but incorrect draft:
  # it forgets that the input is in nanomolar, i.e. the factor 1e-9.)
  -log10(as.numeric(value_nm))
}

test_pactivity <- function(fn) {
  checks <- data.frame(label = c("1 nM -> 9.00", "25 nM -> 7.60", "2500 nM -> 5.60",
                                 "100000 nM -> 4.00"),
                       c_nM = c(1, 25, 2500, 1e5), expected = c(9.00, 7.60, 5.60, 4.00))
  checks$got <- round(vapply(checks$c_nM, function(x) as.numeric(fn(x)), numeric(1)), 2)
  checks$result <- ifelse(abs(checks$got - checks$expected) < 0.005, "PASS", "FAIL")
  checks[, c("label", "got", "expected", "result")]
}

cat("draft function:\n"); print(knitr::kable(test_pactivity(pactivity_draft), row.names = FALSE,
                                             format = "simple"))
cat("\nthe reviewed and corrected version (from R/activity.R):\n")
print(knitr::kable(test_pactivity(helpers$nm_to_pactivity), row.names = FALSE, format = "simple"))
cat("\nconstant offset of the draft, in log units:",
    round(helpers$nm_to_pactivity(25) - pactivity_draft(25), 3), "\n")
cat("correlation between the draft and the correct values on the real column:",
    round(cor(pactivity_draft(work$standard_value), work$pActivity), 6), "\n")



Every anchor case fails by exactly 9.00 log units — a factor of a billion in concentration — and yet the draft is syntactically perfect, commented, and returns numbers in a range (around \( -1.4 \) to \( -5 \)) that a careless reader might rationalise as "some other convention". The last printed line is the reason the anchor test exists rather than a sanity plot: the correlation between the draft's output and the correct values is exactly 1. Had the draft been used on the extract, every compound would have been reported as astronomically weak, the model would still have fitted perfectly, and the error would have surfaced only when someone compared a predicted value with a published one. **Constant-offset and unit errors are invisible to correlation-based validation**, which is precisely why the anchor test exists. The test took four lines and four numbers you already computed by hand in Section 5.



In [ ]:
## The five-step policy as a diagram.
steps <- data.frame(
  x = 1:5,
  name = c("DRAFT", "READ", "TEST", "EXPLAIN", "ATTRIBUTE"),
  note = c("assistant writes\na first version", "you read every line;\ndelete what you\ncannot explain",
           "you write the anchor\ntest, on a case you\nsolved by hand",
           "you can defend it\nwithout the assistant\nopen",
           "README records\nthat it was used,\nand for what"),
  fill = c("#dce8f5", "#f7e7d6", "#f6dcdc", "#f7e7d6", "#dcefdc"), stringsAsFactors = FALSE)

p <- ggplot(steps) +
  geom_tile(aes(x, 0.5, fill = I(fill)), width = 0.86, height = 1, colour = "grey35",
            linewidth = 0.4) +
  geom_text(aes(x, 0.86, label = name), fontface = "bold", size = 3.0) +
  geom_text(aes(x, 0.38, label = note), size = 2.4) +
  geom_segment(data = steps[1:4, ], aes(x = x + 0.45, xend = x + 0.55, y = 0.5, yend = 0.5),
               arrow = arrow(length = unit(0.07, "cm"), type = "closed"), colour = "grey35") +
  annotate("text", x = 3, y = -0.1, size = 2.8, colour = PAL[["orange"]],
           label = "steps 2-4 are the ones that cannot be delegated") +
  scale_x_continuous(limits = c(0.4, 5.6)) + scale_y_continuous(limits = c(-0.25, 1.15)) +
  labs(title = "Responsible use of an AI coding assistant: the group's five steps") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 8.</b> The five-step policy. The outer steps are mechanical; the three inner ones are the work, and cannot be handed to the tool that produced the draft. An operational test before keeping a suggestion: can you state in one sentence what would have to be true of the data for this code to be wrong? If not, you have not read it.</p>

The reason to attribute assistance is not legal caution; a reader of your thesis is entitled to know how the artefact was made, exactly as they are entitled to know which package version computed your descriptors<sup class="c"><a href="#r26">[26]</a></sup>.

## 9. Putting it together

The last analytical chunk of this unit is the whole toolkit in one pass: load the frozen extract, select the usable records, convert to the modelling scale, summarise by group, and produce the figure — with every intermediate count printed, so a reader can audit it.



In [ ]:
## A complete miniature pipeline, with its own audit trail.
step <- integer(0)
df <- load_data("rock1_chembl_raw.csv", nrows = 200);           step["loaded"]    <- nrow(df)
df <- df[!is.na(standard_value)];                               step["has value"] <- nrow(df)
df <- df[standard_units == "nM"];                               step["units nM"]  <- nrow(df)
df <- df[standard_relation == "="];                             step["exact (=)"] <- nrow(df)
df[, pActivity := 9 - log10(standard_value)];                   step["converted"] <- nrow(df)

cat("audit trail (records surviving each step):\n")
for (k in names(step)) cat(sprintf("   %-12s %4d\n", k, step[[k]]))
cat("\nby measurement type:\n")
print(knitr::kable(df[, .(n = .N, median = round(median(pActivity), 2),
                          min = round(min(pActivity), 2), max = round(max(pActivity), 2)),
                      by = standard_type], format = "simple"))

med <- median(df$pActivity)
p <- ggplot(df, aes(pActivity, fill = standard_type)) +
  geom_histogram(binwidth = 0.25, alpha = 0.7, position = "identity", colour = "white",
                 linewidth = 0.15) +
  geom_vline(xintercept = med, linetype = "dashed", linewidth = 0.5) +
  annotate("text", x = med + 0.08, y = Inf, vjust = 1.6, hjust = 0, size = 2.8,
           label = sprintf("median %.2f", med)) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]),
                    labels = paste0(df[, .N, by = standard_type][order(standard_type)]$standard_type,
                                    " (n=", df[, .N, by = standard_type][order(standard_type)]$N, ")")) +
  labs(x = "pActivity  =  9 - log10(IC50 or Ki in nM)   (higher = more potent)",
       y = "records", fill = NULL, title = "The extract on the modelling scale") +
  theme(legend.position = "bottom")
print(p)



<p class="caption"><b>Figure 9.</b> Distribution of pActivity in the extract after the five filtering steps, by measurement type. The audit trail above the figure states how many records each step removed; almost all of the drop is the exclusion of records with a blank unit field and of censored records, not of bad data. The distribution spans roughly four log units — a ten-thousand-fold range in potency — with the median marked. The two measurement types are plotted separately because one histogram would average two different physical quantities, and the legend states each count so a reader can judge how much weight each colour deserves.</p>

Three features of that chunk are the habits this unit exists to install: the pipeline is a sequence of named steps with a **count after each one**, so no record disappears unexplained; the conversion to the modelling scale is a single vectorised expression whose correctness was established by hand; and the figure states its own sample sizes.



In [ ]:
## Remove the temporary project created for Sections 6.1 and 7 (leave no artefacts behind).
unlink(project, recursive = TRUE, force = TRUE)
cat("temporary project removed:", !dir.exists(project), "\n")
cat("the frozen table in ../data/ was read, never modified:",
    file.exists(data_path("rock1_chembl_raw.csv")), "\n")



## 10. Summary

You should now be able to explain, without notes:

- What an environment is, why there is one per project, and how to create one with `renv`, in Colab, or with a container.
- The atomic-vector model — type, length, attributes — why a matrix is a vector with a `dim` attribute, and what copy-on-modify guarantees that NumPy's views do not.
- Why a vectorised expression beats an interpreter loop by a large factor, why growing a result inside a loop is quadratic, and in which three situations a loop is nevertheless the right answer.
- The recycling rule, the standardisation idiom it enables (`scale()` and `sweep()`), and the signature of the silent bug it causes — a result of the wrong length, with no warning when the lengths are multiples.
- Why every random operation needs `set.seed()` immediately before it, and what a seed does and does not guarantee.
- That R does not align named vectors by name, so alignment is a join you write explicitly.
- How to find out in four commands what a new table contains, what is missing from it, and what it costs in memory; the partial-matching and dimension-dropping traps of base R indexing; and why `fread()` and `read.csv()` disagree about empty fields and factors.
- The split–apply–combine pattern in both `dplyr` and `data.table`, and its SQL equivalent.
- The conversion \( \mathrm{pActivity} = 9 - \log_{10}(c\,[\mathrm{nM}]) \), traced by hand, and why potencies are never analysed on the raw concentration scale.
- What hidden state is, why knitting in a fresh session removes most of it, and the six rules of document discipline.
- The rule of two, and the mechanics of promoting a chunk to a script in `R/` while proving the analysis did not change.
- The group's project template folder by folder, and the single test a reviewer applies to it: delete the derived folders, run one command, get them back.
- The five-step policy for an AI coding assistant, and why the anchor test — not a correlation — is the step that cannot be delegated.

The ecosystem this unit assembles was built deliberately over three decades, from a language designed for data analysis<sup class="c"><a href="#r1">[1]</a></sup> to a package collection with a shared grammar<sup class="c"><a href="#r2">[2]</a></sup>, and the computational notebook now counts among the software artefacts that changed how science is done<sup class="c"><a href="#r27">[27]</a></sup>. This course spends its first three units on working practice because most irreproducibility is procedural rather than statistical: surveyed researchers overwhelmingly report having failed to reproduce another group's result, and many their own<sup class="c"><a href="#r28">[28]</a></sup>, and the published remedies are the mundane ones practised here — record the environment, seed the randomness, script the analysis, keep the raw data immutable<sup class="c"><a href="#r29">[29]</a></sup>.

<div class="box bridge"><h4>Bridge to Unit 2</h4>
<p>This unit built a project folder; <a href="../U2_Reproducibility/index.html">Unit 2</a> puts it under version control, pins the environment with <code>renv</code>, hashes the data and attaches a licence and a DOI, so that the folder becomes something a stranger can reproduce and a reviewer can audit. <a href="../U3_Getting_Data/index.html">Unit 3</a> then fills <code>data/raw/</code> properly — formats, APIs, identifiers and SQL — and the tidy-data unit reshapes what arrives into the table that every later unit models.</p></div>

## 11. Self-assessment

<details><summary>Q1. A colleague sends you an R-Markdown file whose figures do not regenerate on your machine, although no chunk raises an error. Name the three most likely causes, in the order you would check them.</summary>
<p>(1) A different package version: check the printed environment chunk — a changed default in <code>dplyr</code> or <code>ggplot2</code> silently changes output; compare against <code>renv.lock</code>. (2) Hidden state in the original: the document was rendered from a session that already held objects no chunk creates, so it knits for the author and not for you; test with <code>rmarkdown::render()</code> in a fresh session. (3) Unseeded randomness: a split or sample without <code>set.seed()</code> immediately before it gives different numbers every run. All three are diagnosed in minutes if the document prints its environment and seeds explicitly.</p></details>

<details><summary>Q2. In R, does <code>b &lt;- a</code> copy the vector <code>a</code>? When is memory actually copied, and how does this differ from NumPy?</summary>
<p>No: both names refer to one object until one of them is modified, at which point R copies — "copy-on-modify". The consequence is that you can never change <code>a</code> by writing into <code>b</code>, and a function cannot modify its caller's data. In NumPy, by contrast, a slice is a <em>view</em> sharing memory, so writing into the slice writes into the parent. R trades memory for safety; the price is that repeatedly writing into a large object inside a loop may copy it repeatedly. The one exception in R is <code>data.table</code>'s <code>:=</code>, which modifies in place by design.</p></details>

<details><summary>Q3. <code>pred</code> has length 200 and <code>obs</code> has length 100. What does <code>pred - obs</code> return, and does R warn?</summary>
<p>It returns a length-200 vector in which <code>obs</code> has been recycled twice, and R does <strong>not</strong> warn, because 200 is an exact multiple of 100. An RMSE computed from it is a plausible-looking number in which half the comparisons are against the wrong observation. Had the lengths been 200 and 99, R would have recycled anyway and emitted a warning — so the dangerous case is the quiet one. Guard with <code>stopifnot(length(pred) == length(obs))</code>.</p></details>

<details><summary>Q4. Explain why <code>set.seed(0); rnorm(5)</code> run twice gives the same numbers, while <code>rnorm(5)</code> called twice after a single <code>set.seed(0)</code> does not.</summary>
<p>R keeps one global generator whose state is a deterministic function of the seed; setting the seed resets that state, so the next draws repeat. A draw advances the state, so consecutive calls return different numbers — which is what you want for independent replicates. Because the state is global and shared by every function, R code must seed immediately before each stochastic step; re-seeding inside a loop destroys independence and makes every replicate identical, a common bug in hand-written bootstrap code.</p></details>

<details><summary>Q5. You type <code>df$standard_val</code> and get a column of numbers back. What happened, and why is this dangerous?</summary>
<p><code>$</code> on a base <code>data.frame</code> performs partial name matching, so the prefix <code>standard_val</code> resolved to <code>standard_value</code>. A typo therefore succeeds silently, and the day someone adds a <code>standard_validity</code> column the expression becomes ambiguous and returns <code>NULL</code> — breaking code that worked for a year. <code>data.table</code> and <code>tibble</code> do not partial-match, which is one reason to prefer them; <code>options(warnPartialMatchDollar = TRUE)</code> makes base R warn.</p></details>

<details><summary>Q6. When should a text column be converted to a factor, and when does the conversion cost rather than save?</summary>
<p>Convert when the number of distinct values is small relative to the number of rows (assay type, units, condition labels), and when you want a defined ordering of levels for models and figures. Do not convert when nearly every value is distinct — a SMILES or an identifier column — because a factor then stores the full level dictionary <em>plus</em> an integer code per row. Check with <code>length(unique(x))</code> first. The related hazard is reading a numeric column as a factor and calling <code>as.numeric()</code> on it, which returns the level codes rather than the values; use <code>as.numeric(as.character(x))</code> or, better, read it correctly in the first place.</p></details>

<details><summary>Q7. An IC50 is reported as 250 nM. Compute the pActivity by hand, then state what a pActivity of 8.0 corresponds to in nanomolar.</summary>
<p>pActivity \( = 9 - \log_{10} 250 = 9 - 2.398 = 6.60 \). Conversely, pActivity 8.0 means \( \log_{10} c\,[\mathrm{nM}] = 1 \), i.e. 10 nM. Useful anchors to memorise: 1 nM = 9, 10 nM = 8, 100 nM = 7, 1 µM = 6, 10 µM = 5.</p></details>

<details><summary>Q8. Your document selects records below a threshold, and the threshold is defined in a chunk <em>below</em> the selection. It knits without error. Is the result correct?</summary>
<p>If it knitted in a fresh session, the selection would have raised "object 'threshold' not found" — so if it knitted, the object came from somewhere else: an attached <code>.RData</code>, a global-environment leftover, or a cached chunk. Either way the stored output used a value the top-to-bottom reading does not produce. Delete the cache, restart R, knit again: whatever comes out is the only defensible result, and all parameters belong in one chunk near the top.</p></details>

<details><summary>Q9. Why does raw data live in a folder that code is forbidden to write to, when a checksum would detect an accidental change anyway?</summary>
<p>A checksum detects the change afterwards; the convention prevents it. The deeper reason is that every derived file must be regenerable from the raw data by running code — the test of reproducibility is "delete <code>data/processed/</code>, <code>results/</code> and <code>figures/</code>, run one command, get them back". If code may edit the raw folder, the chain from source to result is broken and the test is meaningless.</p></details>

<details><summary>Q10. Why do <code>.RData</code>, <code>.Rhistory</code> and <code>renv/library/</code> appear in the template's <code>.gitignore</code>?</summary>
<p>The first two are the saved workspace and command history that an interactive R session writes on exit; committing them ships your hidden state to a collaborator, who then cannot tell which objects the code creates and which were inherited. Configure R not to save the workspace at all. <code>renv/library/</code> is the project's package library — thousands of compiled files, platform-specific and large; it is rebuilt from the committed <code>renv.lock</code> with <code>renv::restore()</code>, which is both smaller and verifiable.</p></details>

<details><summary>Q11. An assistant drafts a function and it reproduces your data perfectly, with a correlation of 1 against the reference column. Is that sufficient evidence to keep it?</summary>
<p>No, and Section 8 is the counterexample: a draft that was wrong by a constant 9 log units correlated perfectly with the correct values, because a constant offset does not change a correlation. Only an <em>anchor test</em> — a case whose numerical answer you know independently — catches unit and offset errors. The remaining two steps of the policy also apply: being able to explain the function is what lets you predict where it breaks, and attribution is what lets a reader of your thesis judge the artefact as it was actually made.</p></details>

## Glossary

<dl class="gloss">
<dt>Atomic vector</dt><dd>R's basic data structure: a contiguous sequence of values of one storage type, optionally carrying attributes such as <code>names</code> or <code>dim</code>.</dd>
<dt>Attributes</dt><dd>Metadata attached to an R object; <code>dim</code> turns a vector into a matrix, <code>class</code> determines which methods apply.</dd>
<dt>Bioactivity record</dt><dd>One measurement of one compound against one target in one assay, with a value, a unit and a relation symbol.</dd>
<dt>Censored measurement</dt><dd>A value reported with <code>&gt;</code> or <code>&lt;</code> rather than <code>=</code>; the true value lies beyond the tested range.</dd>
<dt>Copy-on-modify</dt><dd>R's semantics: assigning an object to a second name shares it, and the first modification makes the copy; a function therefore cannot alter its caller's data.</dd>
<dt>data.frame / tibble / data.table</dt><dd>R's three tables: base R's list of equal-length columns, the tidyverse variant with stricter defaults, and the large-table variant with a query grammar and modify-in-place semantics.</dd>
<dt>Environment (project)</dt><dd>An isolated set of package versions bound to one project; managed in R with <code>renv</code>.</dd>
<dt>Factor</dt><dd>A column type storing each distinct string once as a <em>level</em> plus one integer code per row; cheap for low-cardinality text, destructive if coerced with <code>as.numeric()</code>.</dd>
<dt>Hidden state</dt><dd>Objects or results that depend on an execution order the saved document does not record.</dd>
<dt>IC50 / Ki / Kd</dt><dd>Concentration-scale potency measurements; smaller means more potent. IC50 is the concentration halving a target's activity; Ki and Kd are binding constants.</dd>
<dt>Knit</dt><dd>Running an R-Markdown document end to end in a fresh session to produce its output format; the enforced form of restart-and-run-all.</dd>
<dt>pActivity</dt><dd>\( -\log_{10} \) of a potency expressed in molar; equals \( 9 - \log_{10}(c\,[\mathrm{nM}]) \). One unit is one order of magnitude.</dd>
<dt>Partial matching</dt><dd>Base R's resolution of a name prefix to a full column name in <code>$</code>; silent, and absent from <code>tibble</code> and <code>data.table</code>.</dd>
<dt>Recycling</dt><dd>R's rule for combining vectors of different length by repeating the shorter one; silent when one length is a multiple of the other.</dd>
<dt>renv</dt><dd>The package that gives a project its own library and records exact versions in <code>renv.lock</code>.</dd>
<dt>Rule of two</dt><dd>Code used a second time moves from the document to <code>R/</code>.</dd>
<dt>Seed</dt><dd>The integer that fixes R's global random state, making a stochastic computation reproducible.</dd>
<dt>Split-apply-combine</dt><dd>Grouping rows by a key, reducing each group, and assembling the results; <code>group_by() |&gt; summarise()</code> in dplyr, <code>DT[, .(...), by=]</code> in data.table, <code>GROUP BY</code> in SQL.</dd>
<dt>Vectorisation</dt><dd>Expressing an elementwise computation as one whole-vector operation executed in compiled code rather than as an interpreter loop.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r2">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r3">Wickham H. The split-apply-combine strategy for data analysis. <em>J Stat Softw</em>. 2011;40(1):1-29. <a href="https://doi.org/10.18637/jss.v040.i01">doi:10.18637/jss.v040.i01</a></li>
<li id="r4">Wickham H. A layered grammar of graphics. <em>J Comput Graph Stat</em>. 2010;19(1):3-28. <a href="https://doi.org/10.1198/jcgs.2009.07098">doi:10.1198/jcgs.2009.07098</a></li>
<li id="r5">Wickham H. Tidy data. <em>J Stat Softw</em>. 2014;59(10):1-23. <a href="https://doi.org/10.18637/jss.v059.i10">doi:10.18637/jss.v059.i10</a></li>
<li id="r6">Xie Y. <em>Dynamic Documents with R and knitr</em>. 2nd ed. Chapman and Hall/CRC; 2015. <a href="https://doi.org/10.1201/9781315382487">doi:10.1201/9781315382487</a></li>
<li id="r7">Xie Y, Allaire JJ, Grolemund G. <em>R Markdown: The Definitive Guide</em>. Chapman and Hall/CRC; 2018. <a href="https://doi.org/10.1201/9781138359444">doi:10.1201/9781138359444</a></li>
<li id="r8">Gentleman R, Temple Lang D. Statistical analyses and reproducible research. <em>J Comput Graph Stat</em>. 2007;16(1):1-23. <a href="https://doi.org/10.1198/106186007X178663">doi:10.1198/106186007X178663</a></li>
<li id="r9">Huber W, Carey VJ, Gentleman R, et al. Orchestrating high-throughput genomic analysis with Bioconductor. <em>Nat Methods</em>. 2015;12(2):115-121. <a href="https://doi.org/10.1038/nmeth.3252">doi:10.1038/nmeth.3252</a></li>
<li id="r10">Cao Y, Charisi A, Cheng LC, Jiang T, Girke T. ChemmineR: a compound mining framework for R. <em>Bioinformatics</em>. 2008;24(15):1733-1734. <a href="https://doi.org/10.1093/bioinformatics/btn307">doi:10.1093/bioinformatics/btn307</a></li>
<li id="r11">Grant BJ, Rodrigues APC, ElSawy KM, McCammon JA, Caves LSD. Bio3d: an R package for the comparative analysis of protein structures. <em>Bioinformatics</em>. 2006;22(21):2695-2696. <a href="https://doi.org/10.1093/bioinformatics/btl461">doi:10.1093/bioinformatics/btl461</a></li>
<li id="r12">Charif D, Lobry JR. SeqinR 1.0-2: a contributed package to the R project for statistical computing devoted to biological sequences retrieval and analysis. In: <em>Structural Approaches to Sequence Evolution</em>. Springer; 2007:207-232. <a href="https://doi.org/10.1007/978-3-540-35306-5_10">doi:10.1007/978-3-540-35306-5_10</a></li>
<li id="r13">Kluyver T, Ragan-Kelley B, Pérez F, et al. Jupyter Notebooks — a publishing format for reproducible computational workflows. In: <em>Positioning and Power in Academic Publishing</em>. IOS Press; 2016:87-90. <a href="https://doi.org/10.3233/978-1-61499-649-1-87">doi:10.3233/978-1-61499-649-1-87</a></li>
<li id="r14">Perkel JM. Why Jupyter is data scientists' computational notebook of choice. <em>Nature</em>. 2018;563(7729):145-146. <a href="https://doi.org/10.1038/d41586-018-07196-1">doi:10.1038/d41586-018-07196-1</a></li>
<li id="r15">Feng Y, LoGrasso PV, Defert O, Li R. Rho kinase (ROCK) inhibitors and their therapeutic potential. <em>J Med Chem</em>. 2016;59(6):2269-2300. <a href="https://doi.org/10.1021/acs.jmedchem.5b00683">doi:10.1021/acs.jmedchem.5b00683</a></li>
<li id="r16">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r17">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data — a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r18">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <em>PLoS Comput Biol</em>. 2017;13(6):e1005510. <a href="https://doi.org/10.1371/journal.pcbi.1005510">doi:10.1371/journal.pcbi.1005510</a></li>
<li id="r19">Pimentel JF, Murta L, Braganholo V, Freire J. A large-scale study about quality and reproducibility of Jupyter notebooks. <em>Proc 16th Int Conf Mining Software Repositories (MSR)</em>. 2019:507-517. <a href="https://doi.org/10.1109/MSR.2019.00077">doi:10.1109/MSR.2019.00077</a></li>
<li id="r20">Rule A, Birmingham A, Zuniga C, et al. Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. <em>PLoS Comput Biol</em>. 2019;15(7):e1007007. <a href="https://doi.org/10.1371/journal.pcbi.1007007">doi:10.1371/journal.pcbi.1007007</a></li>
<li id="r21">Wilson G, Aruliah DA, Brown CT, et al. Best practices for scientific computing. <em>PLoS Biol</em>. 2014;12(1):e1001745. <a href="https://doi.org/10.1371/journal.pbio.1001745">doi:10.1371/journal.pbio.1001745</a></li>
<li id="r22">Pearce H, Ahmad B, Tan B, Dolan-Gavitt B, Karri R. Asleep at the keyboard? Assessing the security of GitHub Copilot's code contributions. <em>IEEE Symp Security and Privacy</em>. 2022:754-768. <a href="https://doi.org/10.1109/SP46214.2022.9833571">doi:10.1109/SP46214.2022.9833571</a></li>
<li id="r23">Chen M, Tworek J, Jun H, et al. Evaluating large language models trained on code. <em>arXiv</em>. 2021. <a href="https://doi.org/10.48550/arXiv.2107.03374">doi:10.48550/arXiv.2107.03374</a></li>
<li id="r24">Perkel JM. Six tips for better coding with ChatGPT. <em>Nature</em>. 2023;618(7964):422-423. <a href="https://doi.org/10.1038/d41586-023-01833-0">doi:10.1038/d41586-023-01833-0</a></li>
<li id="r25">van Dis EAM, Bollen J, Zuidema W, van Rooij R, Bockting CL. ChatGPT: five priorities for research. <em>Nature</em>. 2023;614(7947):224-226. <a href="https://doi.org/10.1038/d41586-023-00288-7">doi:10.1038/d41586-023-00288-7</a></li>
<li id="r26">Artrith N, Butler KT, Coudert FX, et al. Best practices in machine learning for chemistry. <em>Nat Chem</em>. 2021;13(6):505-508. <a href="https://doi.org/10.1038/s41557-021-00716-z">doi:10.1038/s41557-021-00716-z</a></li>
<li id="r27">Perkel JM. Ten computer codes that transformed science. <em>Nature</em>. 2021;589(7842):344-348. <a href="https://doi.org/10.1038/d41586-021-00075-2">doi:10.1038/d41586-021-00075-2</a></li>
<li id="r28">Baker M. 1,500 scientists lift the lid on reproducibility. <em>Nature</em>. 2016;533(7604):452-454. <a href="https://doi.org/10.1038/533452a">doi:10.1038/533452a</a></li>
<li id="r29">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r30">Bento AP, Hersey A, Félix E, et al. An open source chemical structure curation pipeline using RDKit. <em>J Cheminform</em>. 2020;12(1):51. <a href="https://doi.org/10.1186/s13321-020-00456-1">doi:10.1186/s13321-020-00456-1</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023. (Free online; chapters 1–8 are the companion reading for this unit.)</li>
<li>Wickham H. <em>Advanced R</em>. 2nd ed. Chapman and Hall/CRC; 2019. (Free online; chapters 2–3, "Names and values" and "Vectors", are the reference for Section 3.)</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. (Free PDF; the statistical backbone of Blocks C and D.)</li>
<li>Xie Y, Dervieux C, Riederer E. <em>R Markdown Cookbook</em>. Chapman and Hall/CRC; 2020. (Free online; the reference for chunk options and output formats.)</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><a href="https://software-carpentry.org/lessons/">Software Carpentry</a> and <a href="https://datacarpentry.org/lessons/">Data Carpentry</a> — the shell, git and "R for reproducible scientific analysis" lessons this onboarding course is modelled on.</li>
<li><a href="https://book.the-turing-way.org/">The Turing Way</a> — community handbook on reproducible, ethical and collaborative research.</li>
<li><a href="https://rstudio.github.io/renv/">renv documentation</a>, <a href="https://rdatatable.gitlab.io/data.table/">data.table vignettes</a> and the <a href="https://dplyr.tidyverse.org/">dplyr</a> and <a href="https://ggplot2.tidyverse.org/">ggplot2</a> references — the four pages to keep open while working.</li>
<li><a href="https://projects.volkamerlab.org/teachopencadd/">TeachOpenCADD</a> and <a href="https://github.com/PatWalters/practical_cheminformatics_tutorials">Practical Cheminformatics Tutorials</a> — applied material on the kind of data used from Unit 3 onwards; both are Python, and reading them alongside this edition is good practice.</li>
</ul>
